In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
print("Installed ✓")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installed ✓


In [2]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("Model + LoRA ready ✓")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.6.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model + LoRA ready ✓


In [3]:
from datasets import load_dataset

dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

def format_prompt(example):
    return {
        "text": f"""### Instruction:
You are a helpful and accurate medical assistant. Answer the patient's question clearly and concisely.

### Patient:
{example['input']}

### Doctor:
{example['output']}<|endoftext|>"""
    }

dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)
dataset = dataset.select(range(5000))
print(f"Dataset ready: {len(dataset)} samples ✓")

Dataset ready: 5000 samples ✓


In [5]:
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
import time

# Free up any leftover memory from previous run
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = 1024,   # ← reduced from 2048 (biggest VRAM saver)
    args = TrainingArguments(
        per_device_train_batch_size  = 1,    # ← reduced from 2
        gradient_accumulation_steps  = 8,    # ← increased to keep effective batch = 8
        warmup_steps                 = 10,
        num_train_epochs             = 1,
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 25,
        output_dir                   = "outputs",
        optim                        = "adamw_8bit",
        save_strategy                = "epoch",
        report_to                    = "none",
    ),
)

print("Starting training... (45–90 mins)")
start = time.time()
trainer.train()
elapsed = round((time.time() - start) / 60, 1)
print(f"\nTraining complete in {elapsed} mins ✓")

Starting training... (45–90 mins)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
25,1.790067
50,1.640725
75,1.597443
100,1.710676
125,1.949251
150,1.920060
175,1.937097
200,1.951853
225,1.942586
250,1.907718


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-625/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-625.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [6]:
# Save LoRA adapter directly — bypasses the checkpoint pickling bug
model.save_pretrained("medical_chatdoctor_lora")
tokenizer.save_pretrained("medical_chatdoctor_lora")
print("Model saved to /content/medical_chatdoctor_lora ✓")

Unsloth: Restored added_tokens_decoder metadata in medical_chatdoctor_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in medical_chatdoctor_lora.


Model saved to /content/medical_chatdoctor_lora ✓


In [7]:
FastLanguageModel.for_inference(model)

prompt = """### Instruction:
You are a helpful and accurate medical assistant. Answer the patient's question clearly and concisely.

### Patient:
I have had a persistent cough for 3 weeks with mild fever. What could this be?

### Doctor:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.split("### Doctor:")[-1].strip())

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Thanks for your question on Chat Doctor. I can understand your concern. Possibility of bronchitis is there in your case. This is inflammatory condition of airways. It is most likely viral infection. You are having persistent cough with mild fever. No need to worry about tuberculosis and lung infection. Just be in contact with your


In [ ]:
from huggingface_hub import login

# Paste your HF write token here
from huggingface_hub import login
login()  # will prompt you to enter token at runtime — never hardcode it
print("Logged in to Hugging Face ✓")

Logged in to Hugging Face ✓


In [12]:
# Replace "your-username" with your actual HF username
HF_USERNAME = "kavya-17b"
REPO_NAME   = "medical-chatdoctor-mistral7b"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"\nModel live at: https://huggingface.co/{HF_USERNAME}/{REPO_NAME} ✓")

README.md:   0%|          | 0.00/577 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

Saved model to https://huggingface.co/kavya-17b/medical-chatdoctor-mistral7b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp_m5thcga/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmp_m5thcga.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p_m5thcga/tokenizer.model: 100%|##########|  493kB /  493kB            


Model live at: https://huggingface.co/kavya-17b/medical-chatdoctor-mistral7b ✓


In [13]:
from huggingface_hub import HfApi

api = HfApi()

model_card = f"""---
language: en
license: apache-2.0
tags:
- medical
- llm
- fine-tuned
- mistral
- lora
- qlora
- unsloth
base_model: mistralai/Mistral-7B-Instruct-v0.2
---

# Medical ChatDoctor — Fine-tuned Mistral 7B

A Mistral-7B model fine-tuned on the [ChatDoctor-HealthCareMagic-100k](https://huggingface.co/datasets/lavita/ChatDoctor-HealthCareMagic-100k) dataset using QLoRA (4-bit quantization) and Unsloth for 2x faster training.

## Training Details
- **Base model**: Mistral-7B-Instruct-v0.2
- **Method**: QLoRA (4-bit) + LoRA rank 8
- **Dataset**: 2,000 medical Q&A samples from ChatDoctor
- **Hardware**: Google Colab free T4 GPU
- **Training time**: ~45 minutes
- **Trainable parameters**: ~42M of 7.2B (0.58%)

## Example Usage
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "{HF_USERNAME}/{REPO_NAME}",
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

prompt = \"\"\"### Instruction:
You are a helpful and accurate medical assistant.

### Patient:
I have had a persistent cough for 3 weeks with mild fever. What could this be?

### Doctor:
\"\"\"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Disclaimer
This model is for educational purposes only. Not a substitute for professional medical advice.
"""

api.upload_file(
    path_or_fileobj = model_card.encode(),
    path_in_repo    = "README.md",
    repo_id         = f"{HF_USERNAME}/{REPO_NAME}",
    repo_type       = "model",
)
print("Model card uploaded ✓")

Model card uploaded ✓


In [14]:
!pip install gradio -q

import gradio as gr
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def medical_chat(patient_question):
    if not patient_question.strip():
        return "Please enter a question."

    prompt = f"""### Instruction:
You are a helpful and accurate medical assistant. Answer the patient's question clearly and concisely.

### Patient:
{patient_question}

### Doctor:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 300,
        temperature    = 0.7,
        do_sample      = True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Doctor:")[-1].strip()

demo = gr.Interface(
    fn          = medical_chat,
    inputs      = gr.Textbox(
                    label       = "Patient Question",
                    placeholder = "e.g. I have had a headache for 2 days with nausea...",
                    lines       = 3
                  ),
    outputs     = gr.Textbox(label="Medical Assistant Response", lines=6),
    title       = "🩺 Medical Q&A — Fine-tuned Mistral 7B",
    description = "Fine-tuned with QLoRA on ChatDoctor dataset. **For educational purposes only — not a substitute for professional medical advice.**",
    examples    = [
        ["I have had a persistent cough for 3 weeks with mild fever. What could this be?"],
        ["I feel dizzy when I stand up quickly. Is this serious?"],
        ["I have been having lower back pain for a week. What should I do?"],
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://86896669da5c7ae2fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
